# RFM Customer Segmentation — Full Analysis
## Business Problem
Marketing budget is limited. The business needs to know:
- Who are the VIP customers worth rewarding?
- Who is about to churn and needs a win-back campaign?
- Where should we NOT spend money?

**RFM** answers this with 3 measurable metrics — no ML required.

| Metric | Formula | Higher = |
|---|---|---|
| **R**ecency | Days since last purchase | More engaged |
| **F**requency | Unique invoice count | More loyal |
| **M**onetary | Total spend (£) | More valuable |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT = Path("..")
IMG  = ROOT / "images"
IMG.mkdir(exist_ok=True)

SNAPSHOT = pd.Timestamp("2024-01-01")  # reference "today" for Recency

df = pd.read_csv(ROOT / "data" / "retail_transactions.csv", parse_dates=["InvoiceDate"])
print(f"Loaded {len(df):,} rows, {df['CustomerID'].nunique():,} customers")

## 1 · Calculate R, F, M

In [ ]:
rfm = (
    df.groupby("CustomerID")
    .agg(
        last_purchase = ("InvoiceDate", "max"),
        frequency     = ("InvoiceNo",   "nunique"),
        monetary      = ("Revenue",     "sum"),
    )
    .reset_index()
)
rfm["recency"] = (SNAPSHOT - rfm["last_purchase"]).dt.days
rfm = rfm.drop(columns="last_purchase")

print(f"RFM table: {rfm.shape}")
print()
print(rfm[["recency","frequency","monetary"]].describe().round(1))

## 2 · Score 1–5 per Metric (Quintiles)

In [ ]:
rfm["R"] = pd.qcut(rfm["recency"],
                   q=5, labels=[5,4,3,2,1]).astype(int)     # lower days → higher score
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"),
                   q=5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"],
                   q=5, labels=[1,2,3,4,5]).astype(int)

rfm["rfm_score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["rfm_total"] = rfm["R"] + rfm["F"] + rfm["M"]

print("Score distributions:")
print(rfm[["R","F","M","rfm_total"]].describe().round(2))
rfm.head(8)

## 3 · Assign Business Segments

In [ ]:
def segment(row):
    r, f, m = row["R"], row["F"], row["M"]
    if r >= 4 and f >= 4:                   return "Champions"
    if r >= 3 and f >= 3:                   return "Loyal Customers"
    if r >= 4 and f <= 2:                   return "New Customers"
    if r <= 2 and f >= 3:                   return "At Risk"
    if r <= 2 and f <= 2 and m >= 3:        return "Cannot Lose Them"
    if r <= 2 and f <= 2:                   return "Lost"
    return "Potential Loyalists"

rfm["segment"] = rfm.apply(segment, axis=1)

summary = (
    rfm.groupby("segment")
    .agg(
        customers     = ("CustomerID", "count"),
        avg_recency   = ("recency",    "mean"),
        avg_frequency = ("frequency",  "mean"),
        avg_monetary  = ("monetary",   "mean"),
        total_revenue = ("monetary",   "sum"),
    )
    .sort_values("total_revenue", ascending=False)
    .round(1)
)
summary["revenue_share_%"] = (summary["total_revenue"] / summary["total_revenue"].sum() * 100).round(1)
print(summary.to_string())

In [ ]:
# Export for Streamlit app
rfm.to_csv(ROOT / "data" / "rfm_scored.csv", index=False)
print("Saved → data/rfm_scored.csv")

## 4 · Visualisations

In [ ]:
COLORS = {
    "Champions":          "#2a9d8f",
    "Loyal Customers":    "#264653",
    "Potential Loyalists":"#e9c46a",
    "New Customers":      "#f4a261",
    "At Risk":            "#e76f51",
    "Cannot Lose Them":   "#c77dff",
    "Lost":               "#adb5bd",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment customer count
seg_n = rfm["segment"].value_counts()
axes[0].barh(seg_n.index, seg_n.values,
             color=[COLORS.get(s,"#888") for s in seg_n.index])
axes[0].set_title("Customers per Segment", fontsize=13)
axes[0].set_xlabel("Customers")

# Segment revenue share
seg_rev = summary["total_revenue"].sort_values()
axes[1].barh(seg_rev.index, seg_rev.values,
             color=[COLORS.get(s,"#888") for s in seg_rev.index])
axes[1].set_title("Total Revenue per Segment (£)", fontsize=13)
axes[1].set_xlabel("Revenue (£)")

plt.tight_layout()
plt.savefig(IMG / "03_segments_overview.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# RFM Scatter — Recency vs Monetary, coloured by segment
fig = px.scatter(
    rfm.sample(min(2000, len(rfm)), random_state=42),
    x="recency", y="monetary",
    color="segment",
    size="frequency",
    size_max=18,
    opacity=0.65,
    color_discrete_map=COLORS,
    hover_data=["CustomerID","frequency","rfm_score"],
    labels={"recency":"Recency (days)","monetary":"Total Spend (£)"},
    title="RFM Scatter — Recency vs Spend  (bubble size = frequency)",
)
fig.update_layout(height=520)
fig.show()

In [ ]:
# Heatmap: average spend by R × F score
heat = rfm.pivot_table(index="R", columns="F", values="monetary", aggfunc="mean")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlGn",
            linewidths=0.4, ax=ax,
            cbar_kws={"label":"Avg Spend (£)"})
ax.set_title("Average Spend by Recency × Frequency Score", fontsize=13)
ax.set_xlabel("Frequency Score (1=low, 5=high)")
ax.set_ylabel("Recency Score  (1=old, 5=recent)")
plt.tight_layout()
plt.savefig(IMG / "04_rf_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 5 · Segment Action Plan

In [ ]:
ACTION = {
    "Champions":           "Reward with early access & exclusive offers. Request reviews.",
    "Loyal Customers":     "Upsell premium categories. Birthday voucher.",
    "Potential Loyalists": "Limited-time offer to trigger 2nd purchase.",
    "New Customers":       "Onboarding email flow. 10% off next order.",
    "At Risk":             "Win-back: 'We miss you — £15 off'. Sense of urgency.",
    "Cannot Lose Them":    "Personal outreach. High-value re-engagement offer.",
    "Lost":                "Last-chance email. Accept churn if no response.",
}

print(f"{'Segment':<22} {'Customers':>9} {'Rev Share':>10}  Marketing Action")
print("─" * 82)
for seg, row in summary.iterrows():
    print(f"{seg:<22} {int(row['customers']):>9,} {row['revenue_share_%']:>9.1f}%  {ACTION.get(seg,'—')}")

## 6 · Business Recommendations

| Priority | Segment | Action | Expected Impact |
|---|---|---|---|
| 🔴 High | At Risk + Cannot Lose Them | Win-back campaign | Recover 15–25% churned revenue |
| 🟡 Medium | Potential Loyalists | 2nd-purchase incentive | +10–15% repeat rate |
| 🟢 Ongoing | Champions | Loyalty programme | Protect top revenue source |
| ⚪ Low | Lost | One email, then stop | Avoid wasted spend |

**RFM refreshes:** re-score every 30–90 days to catch segment movement.